# TissueMC: GPU rung R3 on a Colab T4

Rung R3 keeps R1's per-photon accumulation and changes how threads are used. In R0 and R1 a thread runs one photon to its end, so a warp waits for its longest photon while the other lanes idle (the profiler measured 4.75 of 32 lanes active). In R3 a fixed set of resident threads each claim a photon from a shared counter and **claim the next one as soon as theirs ends**.

This notebook validates R3, compares R1 and R3 with the CPU on identical photons (the comparison now includes absorbed weight, which the R1 comparison did not cover), benchmarks **R1, then R3, then R1 again in the same session** (so the GPU's own drift is visible), and reads the hardware counters for R1 and R3 with Nsight Compute. It ends with **one summary block** for you to paste back.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.*

Then *Runtime → Run all*. It takes roughly 20 to 30 minutes. Nothing here needs passwords or keys.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,clocks.max.sm --format=csv,noheader
!nvcc --version | tail -1

In [ ]:
!git clone -q https://github.com/Raghava-Dwivedi/tissuemc.git
%cd tissuemc
!git log -1 --format='commit %h  %s'

In [ ]:
!pip install -q cmake ninja numpy matplotlib
!pip install -q -e .

In [ ]:
!cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DTISSUEMC_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 > /dev/null
!cmake --build build 2>&1 | tail -3
!./build/tissuemc --version

In [ ]:
# 1. Validate rung 3 against the published values and the independent oracle.
!tissuemc-validate --binary build/tissuemc --engine cuda --rung 3 --report results/gpu-r3-validation.json 2>&1 | grep -v '^  note' | tail -45

In [ ]:
# 2. Same photons through the CPU (float32), rung 1 and rung 3: they should agree far more closely
# than noise, on R_d, T_t and absorbed weight. Rung 1 is included because its absorbed weight was
# only checked through energy conservation before.
!tissuemc-cross-engine --binary build/tissuemc --b-engine cuda --b-rung 1 --photons 2000000 --report results/gpu-r1-cross-engine.json
!tissuemc-cross-engine --binary build/tissuemc --b-engine cuda --b-rung 3 --photons 2000000 --report results/gpu-r3-cross-engine.json

In [ ]:
# 3. Benchmarks, back to back in this session: R1, R3, then R1 again to show session drift.
!tissuemc-bench --engine cuda --rung 1 --binary build/tissuemc --output results/gpu-r1-session.json 2>&1 | tail -4
!tissuemc-bench --engine cuda --rung 3 --binary build/tissuemc --output results/gpu-r3.json 2>&1 | tail -4
!tissuemc-bench --engine cuda --rung 1 --binary build/tissuemc --output results/gpu-r1-again.json 2>&1 | tail -4

In [ ]:
# 4. Hardware counters (Nsight Compute) for both rungs. 2,000,000 photons rather than the default
# 200,000: R3 runs up to about 41,000 resident lanes, so 200,000 photons would give each lane only
# about five photons and the end of the run, when lanes wait for the last long photons, would dominate.
!tissuemc-profile --rung 1 --photons 2000000 --binary build/tissuemc --output results/gpu-r1-profile.json
!tissuemc-profile --rung 3 --photons 2000000 --binary build/tissuemc --output results/gpu-r3-profile.json

In [ ]:
import sys

sys.path.insert(0, "python")
import json
import pathlib
import subprocess

from tissuemc_analysis.ncu_profile import KEY_METRICS


def load(name):
    p = pathlib.Path("results") / name
    return json.loads(p.read_text()) if p.exists() else None


def bench_summary(b):
    if not b:
        return None
    return {
        "median_kernel_s": round(b["kernel"]["median"], 3),
        "kernel_seconds": [round(x, 3) for x in b["kernel_seconds"]],
        "million_events_per_s": round(b["events_per_second"] / 1e6, 1),
        "validated_first": b["validated_before_measuring"],
    }


def cross_summary(c):
    if not c:
        return None
    return {
        "agree": c["agree"],
        "gaps": {g["name"]: [g["value"], g["limit"]] for g in c["gaps"]},
    }


def profile_summary(p):
    if not p:
        return None
    out = {}
    for section, metric, _ in KEY_METRICS:
        entry = p["sections"].get(section, {}).get(metric)
        if entry:
            out[metric] = f"{entry['value']} {entry['unit']}".strip()
    return out


val = load("gpu-r3-validation.json")
cross1, cross3 = load("gpu-r1-cross-engine.json"), load("gpu-r3-cross-engine.json")
commit = subprocess.run(
    ["git", "log", "-1", "--format=%H"], capture_output=True, text=True
).stdout.strip()
r3 = load("gpu-r3.json")
summary = {
    "commit": commit,
    "device": r3["device"] if r3 else None,
    "validation_r3": None
    if not val
    else {
        "checks": len(val),
        "passed": sum(r["passed"] for r in val),
        "failed": [
            f"{r['case']} {r['quantity']} z={r['z']:+.2f} ({r['source']})"
            for r in val
            if not r["passed"]
        ],
    },
    "cross_engine_r1": cross_summary(cross1),
    "cross_engine_r3": cross_summary(cross3),
    "benchmark": {
        "r1_first": bench_summary(load("gpu-r1-session.json")),
        "r3": bench_summary(r3),
        "r1_again": bench_summary(load("gpu-r1-again.json")),
    },
    "counters": {
        "r1": profile_summary(load("gpu-r1-profile.json")),
        "r3": profile_summary(load("gpu-r3-profile.json")),
    },
}
print("===== R3 SUMMARY =====")
print(json.dumps(summary, indent=1))